# Day 13: Capstone Project — Research Paper Q&A Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

---
Your agent demonstrates all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (7 nodes)
2. ✅ ChromaDB RAG (12 documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node with faithfulness scoring + retry loop)
5. ✅ Tool use (DuckDuckGo web search for recent papers)
6. ✅ Deployment (Streamlit UI — app.py)

## My Capstone Plan

**Domain:** Research Paper Q&A for AI/ML  helps students and researchers understand papers

**User:** Students enrolled in AI/ML courses, researchers exploring new topics, professionals upskilling in AI

**Success looks like:** The agent accurately answers questions about AI/ML research papers, cites sources, admits when it doesn't know, and can fetch recent papers via web search

**Tool I will add:** DuckDuckGo Web Search triggered when the user asks about papers published after the knowledge base cutoff or requests paper links/citations

**Deployment choice:** Streamlit UI (`app.py`) accessible chat interface with source citations and faithfulness scores

---
## 0. Setup

**STEP 1:** Make sure you have installed all requirements:
```bash
pip install -r requirements.txt
```

**STEP 2:** Open `.env` and replace `your_gemini_api_key_here` with your real key.
Get a free Gemini API key at: https://aistudio.google.com/app/apikey

In [3]:
# -- Install (uncomment if running in Google Colab) -------------------------
# !pip install langgraph langchain-core langchain-google-genai chromadb \
#              sentence-transformers duckduckgo-search pypdf python-dotenv \
#              streamlit -q

# -- Colab: set Gemini API key ----------------------------------
# from google.colab import userdata
# import os
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

google_key = os.getenv("GOOGLE_API_KEY", "")
print(f"Google Gemini API Key: {'Loaded' if len(google_key) > 10 and google_key != 'your_gemini_api_key_here' else ' Missing — open .env and add your key'}")

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version
import re

print(f"LangGraph version: {version('langgraph')}")


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
r   = llm.invoke("Reply with the single word: ready")
print(f"LLM (Gemini):  {r.content.strip()}")

Google Gemini API Key: Loaded
LangGraph version: 1.1.6
LLM (Gemini):  ready


---
## Part 1 — Domain Setup: Knowledge Base

12 documents covering key AI/ML research paper topics.

**MODIFY:** Add more documents below to expand coverage.
Each document needs `id`, `topic`, and `text` (100-400 words recommended).

In [5]:
# ── Knowledge Base Documents ──────────────────────────────────────────────────
# MODIFY: Add, remove, or edit documents here. Minimum 10 required.

DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "Transformer Architecture",
        "text": """The Transformer architecture, introduced in the paper 'Attention Is All You Need' (Vaswani et al., 2017),
        revolutionised natural language processing by replacing recurrent neural networks with self-attention mechanisms.
        The model consists of an encoder and a decoder, each made up of multiple identical layers. The encoder maps an
        input sequence to a sequence of continuous representations, while the decoder generates the output sequence
        one element at a time. The key innovation is the multi-head self-attention mechanism, which allows the model
        to simultaneously attend to information from different representation subspaces at different positions.
        Positional encodings are added to input embeddings to inject information about the position of tokens in the
        sequence. The Transformer uses three types of attention: encoder self-attention, decoder self-attention, and
        encoder-decoder attention. Feed-forward networks are applied independently to each position. Layer normalisation
        and residual connections are used throughout. The architecture achieves state-of-the-art performance on machine
        translation tasks and became the foundation for models like BERT and GPT."""
    },
    {
        "id": "doc_002",
        "topic": "BERT — Bidirectional Transformers",
        "text": """BERT (Bidirectional Encoder Representations from Transformers), introduced by Devlin et al. in 2018,
        is a pre-training approach for NLP that uses the Transformer encoder. Unlike previous models that read text
        sequentially, BERT reads the entire sequence of words at once, making it deeply bidirectional. It is pre-trained
        on two tasks: Masked Language Modelling (MLM), where 15% of input tokens are masked and the model learns to
        predict them; and Next Sentence Prediction (NSP), where the model learns to understand sentence relationships.
        BERT is fine-tuned on downstream tasks by adding a simple output layer. It achieved state-of-the-art results
        on eleven NLP tasks including question answering, natural language inference, and named entity recognition.
        The base model has 110 million parameters and the large model has 340 million. BERT's bidirectional context
        understanding was a significant advancement over unidirectional models like GPT. It uses WordPiece tokenisation
        and is trained on BooksCorpus and English Wikipedia."""
    },
    {
        "id": "doc_003",
        "topic": "GPT and Autoregressive Language Models",
        "text": """The GPT (Generative Pre-trained Transformer) series, developed by OpenAI, are autoregressive language
        models that generate text by predicting the next token given all previous tokens. GPT-1 (2018) demonstrated
        that language models pre-trained on large text corpora can be fine-tuned for downstream tasks with minimal
        labelled data. GPT-2 (2019) scaled this approach and showed emergent capabilities in text generation, translation,
        and summarisation without task-specific training. GPT-3 (2020) with 175 billion parameters demonstrated few-shot
        and zero-shot learning capabilities across a wide range of tasks using only natural language prompts. GPT-4 (2023)
        is multimodal and outperforms GPT-3 on most benchmarks. These models use a decoder-only Transformer architecture.
        Training involves predicting the next token in a sequence using cross-entropy loss. In-context learning, where
        the model learns from examples provided in the prompt without updating weights, emerged as a surprising property
        of large GPT models. The models are trained on diverse internet text using self-supervised learning."""
    },
    {
        "id": "doc_004",
        "topic": "Retrieval-Augmented Generation (RAG)",
        "text": """Retrieval-Augmented Generation (RAG), introduced by Lewis et al. in 2020, combines parametric memory
        (model weights) with non-parametric memory (external document retrieval) to improve knowledge-intensive NLP tasks.
        In RAG, for each input, relevant documents are retrieved from a large corpus using a dense retrieval model like
        DPR (Dense Passage Retrieval), and then the input and retrieved documents are fed to a seq2seq model (like BART)
        to generate the output. There are two variants: RAG-Sequence uses the same retrieved document for the entire
        output, while RAG-Token can retrieve different documents for each output token. RAG outperforms parametric
        seq2seq models and task-specific retrieval-and-read models on open-domain question answering, abstractive
        question answering, and fact verification tasks. The approach reduces hallucinations because answers must
        be grounded in retrieved evidence. Modern RAG pipelines use vector databases like ChromaDB or FAISS for
        efficient similarity search and large language models as the reader."""
    },
    {
        "id": "doc_005",
        "topic": "Attention Mechanisms",
        "text": """Attention mechanisms allow neural networks to focus on relevant parts of the input when generating
        each part of the output. Bahdanau et al. (2015) introduced the first attention mechanism for neural machine
        translation, enabling the decoder to look back at all encoder hidden states rather than just the final one.
        Scaled dot-product attention, used in Transformers, computes attention scores as: Attention(Q, K, V) =
        softmax(QK^T / sqrt(d_k)) * V, where Q (queries), K (keys), and V (values) are linear projections of the
        input. Multi-head attention runs several attention functions in parallel and concatenates the results,
        allowing the model to attend to information from different representation subspaces. Self-attention relates
        different positions of a single sequence to compute representations of that sequence. Cross-attention relates
        positions in one sequence (decoder) to positions in another (encoder). Sparse attention variants like
        Longformer and BigBird reduce the O(n^2) complexity of full self-attention to handle longer sequences."""
    },
    {
        "id": "doc_006",
        "topic": "Diffusion Models for Image Generation",
        "text": """Diffusion models are a class of generative models that learn to reverse a gradual noising process.
        During training, Gaussian noise is progressively added to data over T timesteps (forward process). The model
        learns to reverse this process by predicting and removing the noise at each step (reverse process). Denoising
        Diffusion Probabilistic Models (DDPM), introduced by Ho et al. (2020), formulated this as a Markov chain and
        showed high-quality image generation. Stable Diffusion uses a latent diffusion model that operates in a
        compressed latent space rather than pixel space, making it computationally efficient. DALL-E 2 and Imagen
        condition the diffusion process on text embeddings using classifier-free guidance to generate images from
        text descriptions. Diffusion models outperform GANs on image quality metrics and offer better mode coverage.
        Applications include image synthesis, inpainting, super-resolution, audio generation, and protein structure
        prediction. The key trade-off is slow sampling speed compared to GANs and VAEs."""
    },
    {
        "id": "doc_007",
        "topic": "Reinforcement Learning from Human Feedback (RLHF)",
        "text": """Reinforcement Learning from Human Feedback (RLHF) is a technique to align language models with
        human preferences. The process has three steps. First, a supervised fine-tuning (SFT) model is trained on
        high-quality demonstration data. Second, a reward model is trained on human preference comparisons — given
        two model outputs, humans rank which is better, and the reward model learns to predict these preferences.
        Third, the SFT model is fine-tuned using proximal policy optimisation (PPO) to maximise the reward model's
        score while staying close to the original model via a KL divergence penalty. This approach was used to
        create InstructGPT and ChatGPT. Direct Preference Optimisation (DPO) is a simpler alternative that bypasses
        the reward model and directly optimises the policy using preference data with a classification loss.
        Challenges include reward hacking, the cost of human labelling, and scalable oversight."""
    },
    {
        "id": "doc_008",
        "topic": "Graph Neural Networks (GNNs)",
        "text": """Graph Neural Networks (GNNs) are deep learning models designed to operate on graph-structured data.
        The core operation is message passing: each node aggregates feature information from its neighbours, applies
        a learnable transformation, and updates its own representation. Graph Convolutional Networks (GCN) by Kipf
        and Welling (2017) use a spectral convolution simplified to first-order neighbourhood aggregation. GraphSAGE
        samples and aggregates features from local neighbourhoods, enabling inductive learning on unseen nodes.
        Graph Attention Networks (GAT) use attention coefficients to weight neighbour contributions. Applications
        include node classification, link prediction, molecular property prediction in drug discovery, social network
        analysis, knowledge graph completion, recommendation systems, and traffic forecasting. A major challenge is
        over-smoothing, where node representations become indistinguishable as the number of layers increases."""
    },
    {
        "id": "doc_009",
        "topic": "Federated Learning",
        "text": """Federated Learning (FL), introduced by McMahan et al. in 2017, is a machine learning paradigm that
        trains models across multiple decentralised devices holding local data without sharing raw data. Each device
        trains a local model and sends only model updates (gradients or weights) to a central aggregator. The aggregator
        combines updates using FedAvg (Federated Averaging), which computes a weighted average of local models. This
        approach preserves data privacy and is useful in healthcare (patient records), mobile devices (keyboard
        predictions), and financial services (fraud detection). Key challenges include statistical heterogeneity
        (non-IID data across clients), system heterogeneity (varying compute and communication capacity), and privacy
        (differential privacy or secure aggregation protect against inference attacks on gradients)."""
    },
    {
        "id": "doc_010",
        "topic": "Contrastive Learning and Self-Supervised Representations",
        "text": """Contrastive learning is a self-supervised representation learning technique that trains a model to
        distinguish similar (positive) pairs from dissimilar (negative) pairs without manual labels. SimCLR (Chen et al.,
        2020) creates two augmented views of each image and trains an encoder to maximise agreement between them using
        NT-Xent loss. MoCo (Momentum Contrast) maintains a dynamic queue of negative samples and uses a momentum
        encoder for consistent representations. CLIP (Contrastive Language-Image Pre-training by OpenAI) aligns image
        and text embeddings in a shared space using contrastive loss trained on 400 million image-text pairs.
        CLIP achieves strong zero-shot transfer to downstream tasks. BYOL and SimSiam learn without negative pairs
        using a bootstrap mechanism. These methods learn rich, transferable features competitive with supervised
        pre-training on many benchmarks."""
    },
    {
        "id": "doc_011",
        "topic": "Neural Architecture Search (NAS)",
        "text": """Neural Architecture Search (NAS) automates the design of neural network architectures. Early NAS
        methods used reinforcement learning (Zoph & Le, 2017) or evolutionary algorithms to search over a large
        architecture space, but required thousands of GPU hours. DARTS (Differentiable Architecture Search) represents
        the search space as a single supernetwork and uses gradient descent to optimise architectural parameters
        jointly with network weights, reducing search cost dramatically. EfficientNet used NAS combined with compound
        scaling to design models achieving state-of-the-art accuracy at various compute budgets. Hardware-aware NAS
        optimises for both accuracy and target hardware metrics like latency and memory footprint. ProxylessNAS and
        Single Path One-Shot methods further reduce memory requirements during search."""
    },
    {
        "id": "doc_012",
        "topic": "LLM Evaluation and Benchmarks",
        "text": """Evaluating Large Language Models (LLMs) requires comprehensive benchmarks covering diverse capabilities.
        GLUE and SuperGLUE benchmark NLU tasks like entailment, coreference resolution, and question answering.
        MMLU (Massive Multitask Language Understanding) tests knowledge across 57 subjects from elementary to
        professional level. HumanEval and MBPP assess code generation by measuring the percentage of problems that
        pass unit tests. BIG-Bench contains hundreds of tasks probing capabilities beyond standard NLP benchmarks.
        TruthfulQA measures whether models generate truthful answers. MT-Bench and Chatbot Arena use GPT-4 or human
        judges to evaluate open-ended conversational quality. RAGAS evaluates RAG pipelines on faithfulness,
        answer relevancy, and context precision. Challenges include benchmark contamination, saturation, and
        difficulty measuring alignment with human values."""
    },
]

# ── Build ChromaDB ────────────────────────────────────────────────────────────
print("Loading embedding model (downloads ~90MB on first run)...")
# MODIFY: Change embedding model name here if needed
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("research_paper_kb")
except:
    pass
collection = client.create_collection("research_paper_kb")

texts      = [d["text"]  for d in DOCUMENTS]
ids        = [d["id"]    for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f" Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   • {d['topic']}")

Loading embedding model (downloads ~90MB on first run)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6265.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Knowledge base ready: 12 documents
   • Transformer Architecture
   • BERT — Bidirectional Transformers
   • GPT and Autoregressive Language Models
   • Retrieval-Augmented Generation (RAG)
   • Attention Mechanisms
   • Diffusion Models for Image Generation
   • Reinforcement Learning from Human Feedback (RLHF)
   • Graph Neural Networks (GNNs)
   • Federated Learning
   • Contrastive Learning and Self-Supervised Representations
   • Neural Architecture Search (NAS)
   • LLM Evaluation and Benchmarks


In [6]:
# ── Test Retrieval ────────────────────────────────────────────────────────────
test_query = "How does attention work in Transformers?"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n Retrieval working correctly.")

Query: How does attention work in Transformers?

Top 3 retrieved chunks:

[1] Topic: Attention Mechanisms
    Text: Attention mechanisms allow neural networks to focus on relevant parts of the input when generating
        each part of the output. Bahdanau et al. (2015) introduced the first attention mechanism for ...

[2] Topic: Transformer Architecture
    Text: The Transformer architecture, introduced in the paper 'Attention Is All You Need' (Vaswani et al., 2017),
        revolutionised natural language processing by replacing recurrent neural networks with...

[3] Topic: BERT — Bidirectional Transformers
    Text: BERT (Bidirectional Encoder Representations from Transformers), introduced by Devlin et al. in 2018,
        is a pre-training approach for NLP that uses the Transformer encoder. Unlike previous model...

 Retrieval working correctly.


---
## Part 2 — State Design

In [7]:
class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:      str          # user's current question

    # ── Memory ─────────────────────────────────────────────
    messages:      List[dict]   # conversation history (sliding window)

    # ── Routing ────────────────────────────────────────────
    route:         str          # "retrieve" | "memory_only" | "tool"

    # ── RAG ────────────────────────────────────────────────
    retrieved:     str          # ChromaDB context chunks
    sources:       List[str]    # source topic names

    # ── Tool ───────────────────────────────────────────────
    tool_result:   str          # DuckDuckGo search output

    # ── Domain-specific ────────────────────────────────────
    paper_title:   str          # extracted paper title (if mentioned in question)

    # ── Answer ─────────────────────────────────────────────
    answer:        str          # final LLM response

    # ── Quality control ────────────────────────────────────
    faithfulness:  float        # eval score 0.0–1.0
    eval_retries:  int          # safety valve counter

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

State defined with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'paper_title', 'answer', 'faithfulness', 'eval_retries']


---
## Part 3 — Node Functions

In [8]:
# ── Node 1: Memory ────────────────────────────────────────────────────────────

def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window — keep last 3 turns
        msgs = msgs[-6:]
    return {"messages": msgs}

# Quick test
result = memory_node({"question": "What is RAG?", "messages": []})
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'What is RAG?'}]
✅ memory_node works


In [9]:
# ── Node 2: Router ────────────────────────────────────────────────────────────
# MODIFY: Update the tool description in the prompt if you change the tool.

def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""You are a router for a Research Paper Q&A chatbot.

Available options:
- retrieve: search the knowledge base for information about AI/ML research papers and concepts
- memory_only: answer from conversation history (e.g. 'what did you just say?', 'can you explain that again?')
- tool: use web search when the user asks about very recent papers (after 2023), author profiles, paper citations, or links

Recent conversation: {recent}
Current question: {question}

Reply with ONLY one word: retrieve / memory_only / tool"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if "memory" in decision:  decision = "memory_only"
    elif "tool" in decision:  decision = "tool"
    else:                     decision = "retrieve"

    # Extract paper title if mentioned in quotes
    title_match = re.search(r'"([^"]+)"|\"([^\"]+)\"', question)
    paper_title = (title_match.group(1) or title_match.group(2)) if title_match else ""

    return {"route": decision, "paper_title": paper_title}

# Quick test
result2 = router_node({"question": "What did you just say?", "messages": [{"role":"user","content":"hi"}]})
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")

router_node test: route='memory_only' (expected: memory_only)


In [10]:
# ── Node 3: Retrieval ─────────────────────────────────────────────────────────

def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}

def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}

# Quick test
result3 = retrieval_node({"question": "How does BERT work?"})
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print(" retrieval_node works")

retrieval_node test: sources=['BERT — Bidirectional Transformers', 'Transformer Architecture', 'Attention Mechanisms']
  Context preview: [BERT — Bidirectional Transformers]
BERT (Bidirectional Encoder Representations from Transformers), introduced by Devlin et al. in 2018,
        is a pre-training approach for NLP that uses the Transf...
 retrieval_node works


In [11]:
# ── Node 4: Tool (DuckDuckGo Web Search) ──────────────────────────────────────
# Triggered for queries about very recent papers or paper links.
# MODIFY: Replace this with any other tool (e.g. Hugging Face Inference API,
#         arXiv API, Wikipedia API, calculator, etc.)

def tool_node(state: CapstoneState) -> dict:
    """Web search for recent papers beyond the knowledge base."""
    question = state["question"]
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(f"research paper {question}", max_results=4))
        if results:
            tool_result = "\n\n".join(
                f"Title: {r.get('title', 'N/A')}\n"
                f"Snippet: {r.get('body', 'N/A')[:300]}\n"
                f"URL: {r.get('href', 'N/A')}"
                for r in results
            )
        else:
            tool_result = "No web results found for this query."
    except ImportError:
        tool_result = "Web search unavailable. Install: pip install duckduckgo-search"
    except Exception as e:
        tool_result = f"Web search error: {str(e)}"

    print(f"  [tool] Web search performed for: {question[:60]}")
    return {"tool_result": tool_result}

print("tool_node defined — DuckDuckGo web search ready")

tool_node defined — DuckDuckGo web search ready


In [12]:
# ── Node 5: Answer ────────────────────────────────────────────────────────────
# MODIFY: Change the system prompt to adjust agent behaviour or strictness.

def answer_node(state: CapstoneState) -> dict:
    question     = state["question"]
    retrieved    = state.get("retrieved", "")
    tool_result  = state.get("tool_result", "")
    messages     = state.get("messages", [])
    eval_retries = state.get("eval_retries", 0)
    paper_title  = state.get("paper_title", "")

    context_parts = []
    if retrieved:   context_parts.append(f"KNOWLEDGE BASE CONTEXT:\n{retrieved}")
    if tool_result: context_parts.append(f"WEB SEARCH RESULTS:\n{tool_result}")
    context = "\n\n".join(context_parts)

    # MODIFY this system prompt for your needs
    if context:
        system_content = f"""You are a knowledgeable Research Paper Q&A assistant specialising in AI and ML research.
Your job is to answer questions about research papers, methodologies, authors, and concepts.

RULES:
1. Answer using ONLY the information in the context below.
2. If the answer is not in the context, say: 'I don't have detailed information about that in my knowledge base.'
3. Cite the topic/paper name when you reference information.
4. If a paper title is mentioned ({paper_title if paper_title else 'none specified'}), focus on that paper.

{context}"""
    else:
        system_content = """You are a helpful Research Paper Q&A assistant.
Answer based on the conversation history. If you cannot answer, say so clearly."""

    if eval_retries > 0:
        system_content += "\n\nIMPORTANT: Answer ONLY from what is explicitly stated in the context above."

    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(
            HumanMessage(content=msg["content"]) if msg["role"] == "user"
            else AIMessage(content=msg["content"])
        )
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}

print("answer_node defined")

answer_node defined


In [13]:
# ── Node 6: Eval (Self-Reflection) ────────────────────────────────────────────

FAITHFULNESS_THRESHOLD = 0.7  # MODIFY: lower = more permissive, higher = stricter
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer  = state.get("answer", "")
    context = state.get("retrieved", "")[:500]
    retries = state.get("eval_retries", 0)

    if not context:
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully grounded. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️ retry"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


# ── Node 7: Save ──────────────────────────────────────────────────────────────
def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}

print("eval_node and save_node defined")

eval_node and save_node defined


---
## Part 4 — Graph Assembly

In [14]:
# ── Routing functions ─────────────────────────────────────────────────────────

def route_decision(state: CapstoneState) -> str:
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"

def eval_decision(state: CapstoneState) -> str:
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry

# ── Build the graph ───────────────────────────────────────────────────────────
graph = StateGraph(CapstoneState)

graph.add_node("memory",   memory_node)
graph.add_node("router",   router_node)
graph.add_node("retrieve", retrieval_node)
graph.add_node("skip",     skip_retrieval_node)
graph.add_node("tool",     tool_node)
graph.add_node("answer",   answer_node)
graph.add_node("eval",     eval_node)
graph.add_node("save",     save_node)

graph.set_entry_point("memory")
graph.add_edge("memory", "router")

graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")

graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print(" Graph compiled successfully!")
print("Flow: memory → router → [retrieve|skip|tool] → answer → eval → save → END")

 Graph compiled successfully!
Flow: memory → router → [retrieve|skip|tool] → answer → eval → save → END


---
## Part 5 — Testing

In [15]:
def ask(question: str, thread_id: str = "test") -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    return app.invoke({"question": question}, config=config)


TEST_QUESTIONS = [
    # Domain questions
    {"q": "What is the Transformer architecture and who introduced it?",      "expect": "Should mention Vaswani et al., encoder-decoder, self-attention",   "red_team": False},
    {"q": "What are the two pre-training tasks used in BERT?",                 "expect": "Should mention MLM and NSP",                                      "red_team": False},
    {"q": "How does GPT-3 perform tasks without fine-tuning?",                 "expect": "Should mention few-shot/zero-shot learning, in-context learning",  "red_team": False},
    {"q": "What problem does RAG solve and how does it reduce hallucinations?", "expect": "Should mention grounding answers in retrieved evidence",           "red_team": False},
    {"q": "Explain how multi-head attention works.",                           "expect": "Should describe parallel attention heads, Q K V",                  "red_team": False},
    {"q": "What is RLHF and what are its three steps?",                       "expect": "Should mention SFT, reward model, PPO",                            "red_team": False},
    {"q": "How are diffusion models different from GANs?",                    "expect": "Should mention noising/denoising process vs adversarial training",  "red_team": False},
    {"q": "What is CLIP and how does it learn visual representations?",       "expect": "Should mention contrastive loss, image-text pairs",                "red_team": False},
    # Red-team tests
    {"q": "What does this agent know about cooking recipes?",                  "expect": "Should admit it doesn't have that information",                    "red_team": True},
    {"q": "Is it true that the Transformer architecture was invented by Google DeepMind in 2015?", "expect": "Should correct the false premise (Google Brain/2017)", "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

Prepared 10 test questions (2 red-team)


In [16]:
test_results = []

print("=" * 60)
print("RUNNING TEST SUITE")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    result = ask(test["q"], thread_id=f"test-{i}")
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:250]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    # PASS if answer is substantive (>30 chars) and for red-team, admits limits
    if test["red_team"]:
        passed = "don't" in answer.lower() or "not" in answer.lower() or "incorrect" in answer.lower() or len(answer) > 30
    else:
        passed = len(answer) > 30

    print(f"Result: {' PASS' if passed else '❌ FAIL'}")
    test_results.append({"q": test["q"][:50], "passed": passed,
                         "faith": faith, "route": route, "red_team": test["red_team"]})

total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")

RUNNING TEST SUITE

--- Test 1  ---
Q: What is the Transformer architecture and who introduced it?
  [eval] Faithfulness: 1.00 ✅
A: The Transformer architecture, introduced by Vaswani et al. in 2017 in the paper 'Attention Is All You Need', revolutionized natural language processing. It replaced recurrent neural networks with self-attention mechanisms.

The Transformer model cons
Route: retrieve | Faithfulness: 1.00
Expected: Should mention Vaswani et al., encoder-decoder, self-attention
Result:  PASS

--- Test 2  ---
Q: What are the two pre-training tasks used in BERT?
  [eval] Faithfulness: 0.50 ⚠️ retry
  [eval] Faithfulness: 0.50 ⚠️ retry
A: BERT is pre-trained on two tasks: Masked Language Modelling (MLM) and Next Sentence Prediction (NSP) (BERT — Bidirectional Transformers).
Route: retrieve | Faithfulness: 0.50
Expected: Should mention MLM and NSP
Result:  PASS

--- Test 3  ---
Q: How does GPT-3 perform tasks without fine-tuning?
  [eval] Faithfulness: 0.00 ⚠️ retry
  [eval] Fait

---
## Part 6 — RAGAS Baseline Evaluation

In [17]:
RAGAS_QUESTIONS = [
    {"question": "What is the Transformer architecture?",
     "ground_truth": "The Transformer, introduced by Vaswani et al. in 2017, uses encoder-decoder structure with multi-head self-attention instead of recurrence."},
    {"question": "What are the two pre-training tasks in BERT?",
     "ground_truth": "BERT uses Masked Language Modelling (MLM) where 15 percent of tokens are masked, and Next Sentence Prediction (NSP) to understand sentence relationships."},
    {"question": "What is RAG and why does it reduce hallucinations?",
     "ground_truth": "RAG combines parametric and non-parametric memory; it retrieves relevant documents and forces the model to ground answers in that evidence."},
    {"question": "What is RLHF and how does it train language models?",
     "ground_truth": "RLHF involves supervised fine-tuning, then training a reward model on human preferences, then PPO to maximise the reward while limiting KL divergence."},
    {"question": "How do diffusion models generate images?",
     "ground_truth": "Diffusion models progressively add Gaussian noise (forward process) and then learn to reverse it step by step (reverse process) to generate images."},
]

eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  ✓ {rq['question'][:55]}")

print(f"\n Eval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation...
  [eval] Faithfulness: 1.00 ✅
  ✓ What is the Transformer architecture?
  [eval] Faithfulness: 0.50 ⚠️ retry
  [eval] Faithfulness: 0.50 ⚠️ retry
  ✓ What are the two pre-training tasks in BERT?
  [eval] Faithfulness: 1.00 ✅
  ✓ What is RAG and why does it reduce hallucinations?
  [eval] Faithfulness: 1.00 ✅
  ✓ What is RLHF and how does it train language models?
  [eval] Faithfulness: 1.00 ✅
  ✓ How do diffusion models generate images?

 Eval dataset built: 5 rows


In [18]:
# Run RAGAS if installed, else fall back to manual faithfulness scoring
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data   = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")
    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )
    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\n  Record these baseline scores in the summary below.")

except ImportError:
    print("RAGAS not installed — running manual faithfulness scoring")
    print("To use RAGAS: pip install ragas datasets")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:45]:45s} → {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nManual Baseline Faithfulness: {avg:.3f}")

RAGAS not installed — running manual faithfulness scoring
To use RAGAS: pip install ragas datasets
  Q: What is the Transformer architecture?         → 1.00
  Q: What are the two pre-training tasks in BERT?  → 0.00
  Q: What is RAG and why does it reduce hallucinat → 0.98
  Q: What is RLHF and how does it train language m → 0.50
  Q: How do diffusion models generate images?      → 0.70

Manual Baseline Faithfulness: 0.636


---
## Part 7 — Deployment

The Streamlit app is already written in `app.py`. Run it with:

```bash
streamlit run app.py
```

It will open at `http://localhost:8501` in your browser.

In [19]:
# Verify app.py exists and is ready
import os
if os.path.exists("app.py"):
    print("✅ app.py exists — ready to deploy")
    print("\nRun this command in your terminal to launch the UI:")
    print("   streamlit run app.py")
    print("\nThe app will open at: http://localhost:8501")
else:
    print("❌ app.py not found — make sure you are in the research_paper_qa/ folder")

✅ app.py exists — ready to deploy

Run this command in your terminal to launch the UI:
   streamlit run app.py

The app will open at: http://localhost:8501


---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Name:** - ROHIT KASAUDHAN

**Roll Number:** - 23053809

**Batch/Program:** - Agentic AI / Course 2026 (Btech- CSE)

**Domain chosen:** Research Paper Q&A for AI/ML

**What the agent does:** This agent answers questions about AI and Machine Learning research papers. It uses a ChromaDB knowledge base of 12 documents covering topics from Transformers to Federated Learning. It routes questions to either RAG retrieval, memory-only answers, or a DuckDuckGo web search tool for recent papers. Conversation memory persists across turns, and a self-reflection eval node checks faithfulness before saving the response.

**Knowledge base:** 12 documents covering: Transformer Architecture, BERT, GPT, RAG, Attention Mechanisms, Diffusion Models, RLHF, Graph Neural Networks, Federated Learning, Contrastive Learning, Neural Architecture Search, and LLM Benchmarks.

**Tool used:** DuckDuckGo web search — triggered when users ask about very recent papers (post-2023), author profiles, or paper citation links that are not in the knowledge base.

**RAGAS baseline scores:**
- Faithfulness: 0.636 (manual baseline, as RAGAS was not fully configured)
- Answer Relevance: Not evaluated
- Context Precision: Not evaluated

**Test results:** 10 / 10 tests passed. Red-team: 2 / 2 passed.

**One thing I would improve with more time:** I would load real PDF papers using PyPDF and chunk them into the knowledge base instead of hand-written summaries, giving the agent access to actual paper content including equations and experiment results.

**Most surprising thing I learned building this:** The router node makes a huge difference — without proper routing, the agent would always hit the retrieval path even for simple follow-up questions, wasting API calls and losing context.

---
## Submission Checklist

- [ ✅] All TODO sections filled in (name, roll number, batch, RAGAS scores)
- [ ✅] Knowledge base has 12 documents 
- [ ✅] All cells run without errors (Kernel → Restart & Run All)
- [ ✅] Test suite shows results for all 10 questions
- [ ✅] RAGAS baseline scores recorded
- [ ✅] `app.py` runs and the chat UI works
- [ ✅] Conversation memory works — tested 3 follow-up questions
- [ ✅] Written summary is complete

**Deliverables:**
1. `day13_capstone.ipynb` — this completed notebook
2. `app.py` — Streamlit deployment
3. `agent.py` — shared agent module
4. `requirements.txt`, `.env` (with key filled in for local use)

---
*Research Paper Q&A Agent 